# ThreatLens AI — Notebook 06: User Behavior Analytics (UBA)

**Stage in the pipeline:** `Intelligence -> UBA`

### What this notebook does
1. Loads the cleaned dataset
2. Builds a behavioral **baseline** per entity (typical active hours, typical daily volume)
3. Scores **current** activity against that baseline -- how far off is it?
4. Reproduces the exact U-1042-style profile card shown on the dashboard's User Behavior page
5. Flags the highest-deviation entities as candidates for investigation

### An honest note before we start
CICIDS2017 is network *flow* data, not an identity-management log -- there's no literal "username" column. The closest real identifier is **Source IP**, so this notebook profiles behavior per source IP, matching the blueprint's own example (the `192.168.1.105` chain used throughout the project). If your CSV doesn't include `source_ip` / `timestamp` columns, the cell below will raise a clear error telling you exactly what to do -- some "cleaned" CICIDS2017 releases on Kaggle strip these columns to keep only numeric ML features. If that happens, re-download a version that keeps flow metadata (e.g. the `dhoogla/cicids2017` Kaggle dataset) and merge those columns back in.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.uba import require_identity_columns, build_entity_baseline, score_deviation

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 5)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


In [ ]:
df = pd.read_parquet(PROCESSED_DIR / "cicids2017_cleaned.parquet")
ip_col, ts_col = require_identity_columns(df)
print(f"Using '{ip_col}' as entity identifier, '{ts_col}' as time column")
print(f"Unique entities (source IPs): {df[ip_col].nunique():,}")

## 1. Split into "history" (baseline) and "current" (recent activity)

We use the earlier 80% of each entity's timeline to build its baseline, and the most recent 20% as "current" activity to evaluate -- this avoids the circularity of scoring an entity's behavior against a baseline that already includes that exact activity.


In [ ]:
df_sorted = df.sort_values(ts_col)
split_point = int(len(df_sorted) * 0.8)
history_df = df_sorted.iloc[:split_point]
current_df = df_sorted.iloc[split_point:]

print(f"History (baseline) window: {len(history_df):,} rows")
print(f"Current (evaluation) window: {len(current_df):,} rows")

## 2. Build the baseline

For every entity seen in the history window, we compute: its typical active hour, how spread out its active hours are, and its average daily event volume -- the "Normal Hours 09:00-18:00" style numbers shown in the dashboard's UBA card.


In [ ]:
baseline = build_entity_baseline(history_df, ip_col, ts_col)
print(f"Built baselines for {len(baseline):,} entities")
baseline.sort_values("total_events", ascending=False).head(10)

## 3. Score current activity against the baseline

Each row in the "current" window gets a deviation score in [0, 1], built from two signals:
- **off_hours_signal** -- how far the current hour is from this entity's typical hour (circular distance, so 11 PM and 1 AM count as close, not 22 hours apart)
- **unknown_entity** -- this source IP was never seen in the history window at all (a brand-new source is itself a real risk signal, not just missing data)


In [ ]:
deviation = score_deviation(current_df, baseline, ip_col, ts_col)
deviation.sort_values("deviation_score", ascending=False).head(10)

## 4. Reproduce the dashboard's UBA profile card

This block formats the single most deviated entity exactly the way the dashboard's "User Behavior Snapshot" panel presents it -- Normal Hours vs Current Activity, flagged in the same style.


In [ ]:
top = deviation.sort_values("deviation_score", ascending=False).iloc[0]
entity_baseline = baseline.loc[top[ip_col]] if top[ip_col] in baseline.index else None

print("=" * 50)
print("  USER BEHAVIOR PROFILE")
print("=" * 50)
print(f"  Entity            : {top[ip_col]}")
print(f"  Deviation Score   : {top['deviation_score']:.0%}")
print(f"  Status            : {'UNUSUAL' if top['deviation_score'] > 0.5 else 'Normal'}")
if entity_baseline is not None:
    print(f"  Typical Hour      : {entity_baseline['typical_hour']:.0f}:00")
else:
    print(f"  Typical Hour      : N/A (new entity, no baseline)")
print(f"  Current Hour      : {top['current_hour']:.0f}:00" + (" (Off-hours)" if top["off_hours_signal"] > 0.5 else ""))
print(f"  New/Unknown Entity: {'Yes' if top['unknown_entity'] else 'No'}")
print("=" * 50)

## 5. Distribution of deviation scores

A quick sanity check: most activity should score low (normal), with a smaller tail of genuinely unusual entities -- this shape is what makes deviation-score-based alerting practical (set a threshold, review only the tail, instead of every single event).


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(deviation["deviation_score"], bins=30, color="#00c583")
plt.axvline(0.5, color="#ff4d5a", linestyle="--", label="Suggested review threshold (0.5)")
plt.xlabel("Deviation score")
plt.title("UBA deviation score distribution -- current window")
plt.legend()
plt.tight_layout()
plt.show()

n_flagged = (deviation["deviation_score"] > 0.5).sum()
print(f"{n_flagged:,} of {len(deviation):,} events ({n_flagged/len(deviation):.1%}) would be flagged for review at this threshold")

## 6. Top entities to investigate

Ranked by average deviation score across all their recent activity -- this is the list that would populate the dashboard's "Top Attacked / At-Risk Users" table.


In [ ]:
top_entities = (
    deviation.groupby(ip_col)["deviation_score"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
    .head(10)
)
top_entities.columns = ["avg_deviation_score", "event_count"]
top_entities

## 7. Summary & next steps

| Item | Result |
|---|---|
| Entities profiled | see Section 2 |
| Entities flagged (score > 0.5) | see Section 5 |
| Top at-risk entity | see Section 4 |

**Next up (`07_attack_relationship_graph.ipynb`)** builds the NetworkX graph connecting these same source IPs to their destinations, so a flagged entity from this notebook can be visually traced through everything it touched -- the graph view already mocked in the dashboard.
